# Benchmarking Gemma4 12B

- The core issue is this: google/gemma-4-12B-it uses a new architecture internally called gemma4_unified. For transformers to load a model, it needs to have that architecture's code built into the library — essentially a registration that says "when you see gemma4_unified, here's how to build and run it."
  
- That registration only landed in the transformers source code on June 3rd, 2026. Stable PyPI releases like 4.51.3 were published long before that, so they simply don't contain that code at all — which is exactly why you get the KeyError: 'gemma4_unified' error. The library sees the model type in the config file, goes to look it up, and finds nothing.

- The source install (git+https://github.com/huggingface/transformers) pulls the absolute latest code directly from GitHub, so it has the gemma4_unified registration. That's the only version that works — not because of a bug or a hack, but because no stable release has been cut yet that includes it.
  
- The tradeoff is that dev builds from source are less tested and can have other rough edges, like the device placement issue you hit. But there's no way around it for this specific model right now. It's just a timing problem.

In [1]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

✅ Done — restart the kernel now


In [2]:
!nvidia-smi

Wed Jul  8 21:36:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   28C    P0            126W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
import sys
sys.path.append("/home/jovyan")

from config_hf_token import HF_TOKEN
from huggingface_hub import login

login(token=HF_TOKEN)

In [4]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()

processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
print(torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free: {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3 MIG 4g.40gb
VRAM total: 42.3 GB
VRAM free: 13.4 GB reserved


In [7]:
from pathlib import Path
# --- Configuration ---
BASE_DIR = Path().resolve()
PROMPT_NAME = "simple"
MODEL_NAME = "gemma4-12b"

folders_to_process = ["correct", "incorrect"]

## QUALITATIVE
#### Baseline - gender neutral- news outlets - correct/incorrect - visible 0s

In [8]:
import sys
from utils.quali_benchmarking import describe_social_media_post_gemma, save_output_txt, output_exists

In [9]:
for folder in folders_to_process:
    image_dir = BASE_DIR / folder
    
    if not image_dir.exists():
        print(f"Directory not found, skipping: {image_dir}")
        continue
    image_files = sorted(image_dir.glob("*.png"))
    if not image_files:
        print(f"No PNG images found in: {image_dir}")
        continue
    print(f"\nProcessing {len(image_files)} image(s) from folder: [{folder.upper()}]")
    for img_path in image_files:
        str_image_path = str(img_path)
    
        if output_exists(str_image_path, folder, BASE_DIR, MODEL_NAME, PROMPT_NAME):
            print(f"⏭ Skipping: {img_path.name}")
            continue
    
        result = describe_social_media_post_gemma(str_image_path, model, processor, device)
        save_output_txt(str_image_path, result, folder_name=folder, base_dir=BASE_DIR, model_name=MODEL_NAME, prompt_name=PROMPT_NAME)

print("\nAll folders processed successfully!")


Processing 4 image(s) from folder: [CORRECT]
⏭ Skipping: 001_fox_news_c.png
⏭ Skipping: 001_ny_times_c.png
⏭ Skipping: 001_remy_ashford_c.png
⏭ Skipping: 001_reuters_c.png

Processing 4 image(s) from folder: [INCORRECT]
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/qualitative/incorrect/001_fox_news_i_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/qualitative/incorrect/001_ny_times_i_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/qualitative/incorrect/001_remy_ashford_i_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/qualitative/incorrect/001_reuters_i_simple.txt

All folders processed successfully!


### Analysis

### Incorrect posts - 3 of them addressed the incorrect claim after breaking down the visualization
#### 1. Fox News:

#### 2. The New York Times


#### 3. Remy Ashford


### Incorrect posts - Reuters, the final post provided, included a remark about the post claim being wrong within the analysis of the pie chart, instead of adding a statement at the end
#### 4. Reuters


## QUANTITATIVE

**Tests 1-3** establish a baseli: — how well can the model read charts and verify claims when there are no social signals present.

**Test 4** introduces social signals (reaction metrics) and : s — does the model's claim verification accuracy change when the post appears highly liked, or highly reacted to with angry/sad emotiesis scope?

## 1) Extensive analysis of all baseline examples (1 gender neutral user - 3 news outlets) 

In [10]:
import sys
from pathlib import Path
from utils.quanti_benchmarking_1_details import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question_gemma
)

In [11]:
# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"
ASK_FN = ask_question_gemma
variants = ["correct", "incorrect"]

all_images = []
for variant_folder in variants:
    for png in sorted((BASE_DIR / variant_folder).glob("*.png")):
        all_images.append((png.stem, str(png)))

for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning two-call prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

print("\n✅ All versions complete.")


Running prompt version: v1
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_fox_news_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_ny_times_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_remy_ashford_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_reuters_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_fox_news_i.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/gemma4-12b/quantitative/test-1-gn-news-extensive/v1/001_ny_times_i.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarki

KeyboardInterrupt: 

### Accuracy calculations for each version

In [ ]:
from utils.quanti_benchmarking_1_analysis import run_accuracy_analysis

BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"

run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

## Takeaways
-  Gemma simply isn't sensitive to that phrasing variation for this task
-  it has however **clear issues with reading the value % for Pop (0% accuracy each time)**
-  **update on pop %**: after running test 3 i realized quite often despite asking for simply the number gemma still provides as answer "23.5%", which would explain why it is always getting that question wrong!
-  
-  Gemma is just producing a stable, repeatable description-style answer rather than actually responding to subtle prompt differences in your PROMPT_VERSIONS dict (v1 through v6 are all phrased differently but ask for the same "correct"/"incorrect" judgment).
-  If Gemma reliably reaches the same correct/incorrect conclusion regardless of phrasing, that's not necessarily a bug — it could mean Gemma is robust to prompt rewording in a way Qwen wasn't, since do_sample=False makes generation fully deterministic and the underlying image+task is the same across v1-v6.

## 2) Focusing on gender neutral user - using best performing prompt - only asking wether claim is correct or not - 100 versions of visualization remy ashford - 50/50 correct incorrect ratio - store the order provided to the LLM
After part 1, should definitely be using the two call approach here.


In [ ]:
# Unzipping correct and incorrect versions of gender neutral baseline with 100 different visualizations each
#!unzip correct/remy-ashford/remy_ashford_c_pngs.zip -d correct/remy-ashford/
#!unzip incorrect/remy-ashford/remy_ashford_i_pngs.zip -d incorrect/remy-ashford/

In [ ]:
import sys
import random
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_2_claim_only import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question_gemma
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
ASK_FN = ask_question_gemma
SEED = 42
SAMPLE_SIZE = 50

correct_dir = BASE_DIR / "correct/remy-ashford"
incorrect_dir = BASE_DIR / "incorrect/remy-ashford"
all_numbers = sorted([p.name.split("_")[0] for p in correct_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {correct_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
all_images = []
for num in selected_numbers:
    all_images.append((f"{num}_correct", str(correct_dir / f"{num}_remy_ashford_c.png")))
    all_images.append((f"{num}_incorrect", str(incorrect_dir / f"{num}_remy_ashford_i.png")))
print(f"Selected {len(selected_numbers)} pairs → {len(all_images)} images total")

for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

print("\n✅ All remy-ashford versions complete.")

In [ ]:
from utils.quanti_benchmarking_2_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

## 3) Ask about nr in pie charts - both genres - 100 - 50/50
- Reusing all_images from prevous benchmark, in other words, using the same 50 pairs used for previous benchmark

**Why**
- Direct comparability: if test 2 (claim verification) and test 3 (percentage reading) use the same images, we can link results. For example: "the model read the percentages correctly on image 042 but still got the claim wrong": that's a meaningful finding about where reasoning breaks down.
- Controls for image variability: if the sets differ, a performance difference between tests could be due to one set happening to have easier images rather than the task itself being easier.
- Cleaner narrative 


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_3_percentages import (
    benchmark_image_percentages, output_exists_json, ask_question_gemma
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
MODEL_NAME = "gemma4-12b"
ASK_FN = ask_question_gemma

# --- Run ---
print(f"\n{'='*60}")
print("Running test-3: percentage extraction")
print(f"{'='*60}")
for image_name, image_path in all_images:
    if output_exists_json(image_name, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
        print(f"⏭ Skipping: {image_name}")
        continue
    benchmark_image_percentages(image_path, image_name, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
print("\n✅ Test 3 complete.")

In [ ]:
from utils.quanti_benchmarking_3_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

- Issue here is that quite often the model is providing the % symbol too which immediatly marks the answwer as wrong
- However there is more variation in accuracy for pop than for latin, for latin read the same value regardless of correct/incorrect, wih¿th pop it performed better with incorrect variants...

## old 4) Nr of reactions - cover all reactions - equally distributed (will probably keep out because unrealistic)
Does the overall volume of engagement metrics influence the model's claim verification accuracy, independent of reaction type distribution?

By keeping all reactions equal we are controlling for reaction type bias because the model can't be swayed by seeing mostly angry vs mostly love reactions. 

In [ ]:
#!unzip correct/remy-ashford/correct_uniform.zip -d correct/remy-ashford/metrics/uniform
#!unzip incorrect/remy-ashford/incorrect_uniform.zip -d incorrect/remy-ashford/metrics/uniform

The same 50 numbers are reused across all 6 scale values, giving 600 images total per prompt version (50 pairs × 6 scale values × 2 variants)

In [ ]:
# codigo viejo incluye two-call approach y se encuentra en copia de este notebook

# 4) Nr of reactions - more realistic -  X total across all reaction types (likes + loves + hahas etc. combined), x being the log numebr so 10, 100, 1000, etc.
Posts were generated under two reaction conditions: uniform, in which all reaction types were set equally, and realistic, in which reactions were distributed using log-scaled weights to approximate empirical engagement patterns on social media.

With uniform reactions, every post at scale_value=1000 showed exactly 1000 likes, 1000 loves, 1000 hahas etc. — which is something that essentially never occurs on real Facebook and could itself be a signal to the VLM that something artificial is happening. The realistic condition removes that artificiality while keeping scale_value as a clean, interpretable independent variable representing **total engagement volume**.

**Why Jitter Matters**
Without jitter, every image index at a given scale_value would produce identical reaction counts. For example, at scale_value=1000 every single one of your 100 images would show:


- Emoji order is always like, love, haha, wow, sad, angry — fixed by the change in the .py file
- Values vary across images because shares are shuffled using seed=i — so image 001 always gets the same distribution but different from image 002
- Total reactions always sum to approximately scale_value — Interpretation 1
- Files land in realistic/ keeping them separate from your uniform/ condition
- Reproducible — rerunning will generate identical files

In [ ]:
#!unzip correct/remy-ashford/correct_realistic.zip -d correct/remy-ashford/metrics/realistic
#!unzip incorrect/remy-ashford/incorrect_realistic.zip -d incorrect/remy-ashford/metrics/realistic

In [ ]:
import sys
import random
from pathlib import Path
from utils.quanti_benchmarking_4_reactions import (
    PROMPT_VERSIONS, benchmark_image, output_exists_json, ask_question_gemma
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"
ASK_FN = ask_question_gemma
SEED = 42
SAMPLE_SIZE = 50
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]

# --- Build paired sample ---
correct_base = BASE_DIR / "correct/remy-ashford/metrics/realistic"
incorrect_base = BASE_DIR / "incorrect/remy-ashford/metrics/realistic"
sample_dir = correct_base / "10"
all_numbers = sorted([p.name.split("_")[0] for p in sample_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {sample_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
print(f"Selected {len(selected_numbers)} numbers: {selected_numbers}")

all_images = []
for scale_value in REACTION_VALUES:
    for num in selected_numbers:
        all_images.append((
            f"{num}_correct_{scale_value}",
            str(correct_base / str(scale_value) / f"{num}_remy_ashford_c.png")
        ))
        all_images.append((
            f"{num}_incorrect_{scale_value}",
            str(incorrect_base / str(scale_value) / f"{num}_remy_ashford_i.png")
        ))
print(f"Total images to process: {len(all_images)}")

# --- Run ---
for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

print("\n✅ All metrics versions complete.")

In [ ]:
from utils.quanti_benchmarking_4_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)